In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("airline_data_processing")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.executor.cores", 2)
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.dynamicAllocation.maxExecutors", 5)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.warehouse.dir", "hdfs://master/user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

# Verify it worked
# spark.conf.get("spark.sql.catalogImplementation")  # should return "hive"

# # List HDFS root
# files = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem \
#     .get(spark.sparkContext._jsc.hadoopConfiguration()) \
#     .listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path("/"))

# for f in files:
#     print(f.getPath())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 17:51:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [54]:
# Checking configurations
all_configs = spark.sparkContext.getConf().getAll()

# Print the configurations
for config_item in sorted(all_configs):
    print(config_item)

('hive.metastore.uris', 'thrift://master:9083')
('spark.app.id', 'app-20260228040519-0002')
('spark.app.name', 'airline_exploration')
('spark.app.startTime', '1772251519620')
('spark.app.submitTime', '1772184290331')
('spark.driver.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNN

## Data loading

In [23]:
from pyspark.sql.types import *

schema = StructType([
    StructField("FL_DATE",             StringType(),  True),
    StructField("OP_CARRIER",          StringType(),  True),
    StructField("OP_CARRIER_FL_NUM",   StringType(),  True),
    StructField("ORIGIN",              StringType(),  True),
    StructField("DEST",                StringType(),  True),
    StructField("CRS_DEP_TIME",        StringType(),  True),
    StructField("DEP_TIME",            StringType(),  True),
    StructField("DEP_DELAY",           FloatType(),   True),
    StructField("TAXI_OUT",            FloatType(),   True),
    StructField("WHEELS_OFF",          StringType(),  True),
    StructField("WHEELS_ON",           StringType(),  True),
    StructField("TAXI_IN",             FloatType(),   True),
    StructField("CRS_ARR_TIME",        StringType(),  True),
    StructField("ARR_TIME",            StringType(),  True),
    StructField("ARR_DELAY",           FloatType(),   True),
    StructField("CANCELLED",           IntegerType(), True),
    StructField("CANCELLATION_CODE",   StringType(),  True),
    StructField("DIVERTED",            IntegerType(), True),
    StructField("CRS_ELAPSED_TIME",    FloatType(),   True),
    StructField("ACTUAL_ELAPSED_TIME", FloatType(),   True),
    StructField("AIR_TIME",            DoubleType(),  True),
    StructField("DISTANCE",            DoubleType(),  True),
    StructField("CARRIER_DELAY",       DoubleType(),  True),
    StructField("WEATHER_DELAY",       DoubleType(),  True),
    StructField("NAS_DELAY",           DoubleType(),  True),
    StructField("SECURITY_DELAY",      DoubleType(),  True),
    StructField("LATE_AIRCRAFT_DELAY", DoubleType(),  True),
])

In [24]:
full_flight_data = spark.read.csv(
    "hdfs:///user/jupyter/airline_delay_analysis/*.csv",
    header=True,
    schema=schema
)

In [5]:
full_flight_data.count()

71526575

In [25]:
# These columns are mostly null
variables_to_remove = ['TAXI_OUT', 'TAXI_IN', 'WHEELS_ON', 'WHEELS_OFF', 'CANCELLED', 'CANCELLATION_CODE','DIVERTED',
                       'CRS_ELAPSED_TIME','ACTUAL_ELAPSED_TIME','AIR_TIME','DISTANCE','CARRIER_DELAY','WEATHER_DELAY',
                       'NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY','Unnamed: 27']
full_flight_data = full_flight_data.drop(*variables_to_remove)

In [26]:
len(full_flight_data.columns)

11

In [27]:
# Save processed file as parquet for faster loading in the future
full_flight_data.write.mode("overwrite").parquet("hdfs:///user/jupyter/processed/airline_parquet")

In [28]:
# remove the old raw dataset from memory
spark.catalog.clearCache()

In [29]:
processed_flight_data = spark.read.parquet("hdfs:///user/jupyter/processed/airline_parquet").cache()
processed_flight_data.head(5)

[Row(FL_DATE='2019-07-03', OP_CARRIER='OO', OP_CARRIER_FL_NUM='5056', ORIGIN='DLH', DEST='ORD', CRS_DEP_TIME='1840', DEP_TIME='10.00', DEP_DELAY=25.0, CRS_ARR_TIME='22.00', ARR_TIME='76.00', ARR_DELAY=397.0),
 Row(FL_DATE='2019-07-03', OP_CARRIER='OO', OP_CARRIER_FL_NUM='5057', ORIGIN='DEN', DEST='EAR', CRS_DEP_TIME='2032', DEP_TIME='1.00', DEP_DELAY=22.0, CRS_ARR_TIME='-2.00', ARR_TIME='46.00', ARR_DELAY=305.0),
 Row(FL_DATE='2019-07-03', OP_CARRIER='OO', OP_CARRIER_FL_NUM='5058', ORIGIN='DEN', DEST='RKS', CRS_DEP_TIME='1611', DEP_TIME='56.00', DEP_DELAY=23.0, CRS_ARR_TIME='54.00', ARR_TIME='42.00', ARR_DELAY=260.0),
 Row(FL_DATE='2019-07-03', OP_CARRIER='OO', OP_CARRIER_FL_NUM='5059', ORIGIN='CGI', DEST='PAH', CRS_DEP_TIME='1646', DEP_TIME='1.00', DEP_DELAY=11.0, CRS_ARR_TIME='-19.00', ARR_TIME='15.00', ARR_DELAY=45.0),
 Row(FL_DATE='2019-07-03', OP_CARRIER='OO', OP_CARRIER_FL_NUM='5059', ORIGIN='ORD', DEST='CGI', CRS_DEP_TIME='1442', DEP_TIME='-3.00', DEP_DELAY=36.0, CRS_ARR_TIME='1

In [30]:
processed_flight_data.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: float (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: float (nullable = true)



In [19]:
spark.sql("show tables").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|processed_flight_...|      false|
|  default|                 src|      false|
+---------+--------------------+-----------+



In [35]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS flights_clean (
  FL_DATE             STRING,
  OP_CARRIER          STRING,
  OP_CARRIER_FL_NUM   STRING,
  ORIGIN              STRING,
  DEST                STRING,
  CRS_DEP_TIME        STRING,
  DEP_TIME            STRING,
  DEP_DELAY           FLOAT,
  CRS_ARR_TIME        STRING,
  ARR_TIME            STRING,
  ARR_DELAY           FLOAT
)
STORED AS PARQUET
LOCATION '/user/jupyter/processed/airline_parquet'
""")

DataFrame[]

In [36]:
df_5 = spark.sql("SELECT * FROM flights_clean limit 5")
df_5.toPandas()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,CRS_ARR_TIME,ARR_TIME,ARR_DELAY
0,2019-07-03,OO,5056,DLH,ORD,1840,10.00,25.0,22.00,76.00,397.0
1,2019-07-03,OO,5057,DEN,EAR,2032,1.00,22.0,-2.00,46.00,305.0
2,2019-07-03,OO,5058,DEN,RKS,1611,56.00,23.0,54.00,42.00,260.0
3,2019-07-03,OO,5059,CGI,PAH,1646,1.00,11.0,-19.00,15.00,45.0
4,2019-07-03,OO,5059,ORD,CGI,1442,-3.00,36.0,12.00,58.00,339.0


In [5]:
spark.stop()